# Task 3.2: Failure Mode Analysis
**Student:** Bhavishya Sahay | Roll No: 230071

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from scipy.optimize import linear_sum_assignment
from scipy.special import logsumexp
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
K = 3; P_NEIGHBORS = 10; LAMBDA = 0.1; MAX_ITER = 100; TOL = 1e-4; REG = 1e-4

data = load_wine()
X_raw, y_true = data.data, data.target
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X_raw)

def clustering_accuracy(y_true, y_pred):
    n = max(y_true.max(), y_pred.max()) + 1
    cost = np.zeros((n, n), dtype=int)
    for t, p in zip(y_true, y_pred):
        cost[t][p] += 1
    row, col = linear_sum_assignment(-cost)
    return cost[row, col].sum() / len(y_true)

def build_knn_graph(X, p=P_NEIGHBORS):
    from sklearn.neighbors import kneighbors_graph
    G = kneighbors_graph(X, n_neighbors=p, mode='connectivity', include_self=False).toarray()
    return np.maximum(G, G.T)

def log_gaussian(x, mu, Sigma):
    d = len(mu)
    diff = x - mu
    try:
        sign, logdet = np.linalg.slogdet(Sigma)
        if sign <= 0: return -1e10
        return -0.5*(d*np.log(2*np.pi) + logdet + diff @ np.linalg.inv(Sigma) @ diff)
    except: return -1e10

def run_lcgmm(X, K=K, p=P_NEIGHBORS, lam=LAMBDA, use_knn=True, use_kl=True, max_iter=MAX_ITER, tol=TOL, reg=REG):
    N, D = X.shape
    W = build_knn_graph(X, p) if use_knn else np.zeros((N,N))
    rows, cols = np.where(W > 0)
    gmm_init = GaussianMixture(n_components=K, random_state=SEED, reg_covar=reg)
    gmm_init.fit(X)
    pi = gmm_init.weights_.copy(); mu = gmm_init.means_.copy()
    Sigma = gmm_init.covariances_.copy() + reg*np.eye(D)
    prev_obj = -np.inf
    for it in range(max_iter):
        log_resp = np.zeros((N,K))
        for k in range(K):
            for i in range(N):
                log_resp[i,k] = np.log(pi[k]+1e-300) + log_gaussian(X[i], mu[k], Sigma[k])
        log_norm = logsumexp(log_resp, axis=1, keepdims=True)
        R = np.exp(log_resp - log_norm)
        Nk = R.sum(0)+1e-10; pi = Nk/N
        for k in range(K):
            rk = R[:,k]
            mu_std = (rk[:,None]*X).sum(0)/Nk[k]
            reg_mu = np.zeros(D)
            for i,j in zip(rows,cols):
                if use_kl: reg_mu += (rk[i]-rk[j])*(X[i]-X[j])
                else:
                    w_ij = np.linalg.norm(R[i]-R[j])
                    reg_mu += w_ij*(X[i]-X[j])*np.sign(rk[i]-rk[j]+1e-10)
            mu[k] = mu_std - lam*reg_mu/(2*Nk[k])
            diff = X-mu[k]
            S_std = (rk[:,None,None]*diff[:,:,None]*diff[:,None,:]).sum(0)/Nk[k]
            reg_S = np.zeros((D,D))
            for i,j in zip(rows,cols):
                Si = np.outer(X[i]-mu[k],X[i]-mu[k]); Sj = np.outer(X[j]-mu[k],X[j]-mu[k])
                if use_kl: reg_S += (rk[i]-rk[j])*(Si-Sj)
                else:
                    w_ij = np.linalg.norm(R[i]-R[j])
                    reg_S += w_ij*(Si-Sj)*np.sign(rk[i]-rk[j]+1e-10)
            Sigma[k] = S_std - lam*reg_S/(2*Nk[k]) + reg*np.eye(D)
        obj = log_norm.sum()
        if abs(obj-prev_obj)<tol: break
        prev_obj=obj
    return R.argmax(1)

print("Setup complete.")

Setup complete.


## Failure Scenario: Over-Regularisation and Absence of Manifold Structure

**Scenario 1 (lambda sensitivity):** The method fails when the regularisation parameter λ is too large. LCGMM's objective (Equation 6) is a trade-off between log-likelihood and the KL-regulariser term. When λ >> 0.1, the regulariser dominates, forcing the algorithm to assign all connected graph nodes the same posterior regardless of their actual Gaussian likelihood, which collapses meaningful cluster distinctions.

**Scenario 2 (uniform noise):** The method also fails when data is drawn uniformly from a high-dimensional space with no manifold structure. The kNN graph will connect random pairs of points, and the regulariser will penalise correct cluster assignments for boundary points — both LCGMM and GMM revert to near-chance accuracy, but LCGMM's regulariser adds additional noise.

**Why I expected failure:** These scenarios directly violate **Assumption 1 from Task 1.2** — the manifold assumption — and the hyperparameter sensitivity follows directly from the objective function (Eq. 6).


In [2]:
# ============================================================
# FAILURE MODE: Over-regularisation (high lambda)
# ============================================================
lambdas = [0, 0.001, 0.01, 0.1, 0.5, 1.0, 2.0]
lambda_accs = []
for lam in lambdas:
    if lam == 0:
        gm = GaussianMixture(n_components=K, random_state=SEED)
        gm.fit(X)
        acc = clustering_accuracy(y_true, gm.predict(X))
    else:
        lbl = run_lcgmm(X, lam=lam)
        acc = clustering_accuracy(y_true, lbl)
    lambda_accs.append(acc*100)
    print(f"  lambda={lam}: {acc:.1f}%")

  lambda=0: 1.0%
  lambda=0.001: 1.0%
  lambda=0.01: 1.0%
  lambda=0.1: 0.9%
  lambda=0.5: 0.4%
  lambda=1.0: 0.4%
  lambda=2.0: 0.4%


The lambda sweep runs LCGMM with λ ∈ {0, 0.001, 0.01, 0.1, 0.5, 1.0, 2.0}. λ=0 is equivalent to pure GMM. The uniform noise experiment creates 200 random 20-dimensional points with 4 equal-sized classes, designed to have no manifold structure.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
ax = axes[0]
ax.plot([str(l) for l in lambdas], lambda_accs, 'o-', color='#2196F3', linewidth=2.5, markersize=9)
ax.axhline(y=lambda_accs[0], color='green', linestyle='--', linewidth=1.5, label=f'GMM (λ=0): {lambda_accs[0]:.1f}%')
ax.fill_between(range(len(lambdas)), [lambda_accs[0]]*len(lambdas), lambda_accs,
                alpha=0.15, color='red', where=[a < lambda_accs[0] for a in lambda_accs])
ax.set_xlabel('Regularisation Parameter λ', fontsize=12)
ax.set_ylabel('Clustering Accuracy (%)', fontsize=12)
ax.set_title('Failure Mode: Over-Regularisation\n(λ sensitivity on Wine dataset)', fontsize=11, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3); ax.set_ylim(30, 105)

# Failure on uniform noise data
np.random.seed(SEED)
X_fail = np.random.uniform(-3, 3, (200, 20))
y_fail = np.repeat([0,1,2,3], 50)
fail_lbl = run_lcgmm(X_fail, K=4, p=10, lam=0.1)
fail_acc = clustering_accuracy(y_fail, fail_lbl)
gm_fail = GaussianMixture(n_components=4, random_state=SEED)
gm_fail.fit(X_fail)
gmm_fail_acc = clustering_accuracy(y_fail, gm_fail.predict(X_fail))

ax2 = axes[1]
bars = ax2.bar(['LCGMM','GMM'], [fail_acc*100, gmm_fail_acc*100],
               color=['#2196F3','#4CAF50'], edgecolor='black', linewidth=0.8, width=0.3)
ax2.axhline(y=25, color='red', linestyle='--', label='Chance (25%)')
ax2.set_ylim(0, 60)
ax2.set_ylabel('Clustering Accuracy (%)', fontsize=12)
ax2.set_title('Failure Mode: Uniform High-Dim Data\n(No Manifold Structure)', fontsize=11, fontweight='bold')
for bar, acc in zip(bars, [fail_acc*100, gmm_fail_acc*100]):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{acc:.1f}%', ha='center', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
plt.tight_layout()
plt.savefig('/Users/bhavishyamac/Desktop/230071-midesm/partB/results/task3_failure.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Failure mode: LCGMM={fail_acc*100:.1f}%, GMM={gmm_fail_acc*100:.1f}%")
print("Saved to partB/results/task3_failure.png")

Failure mode: LCGMM=31.0%, GMM=29.5%
Saved to partB/results/task3_failure.png


## Explanation of Failure

**Over-regularisation (λ > 0.1):** Accuracy collapses from 93.8% to ~39.9% when λ reaches 0.5, and remains near 40% (essentially assigning all points to one or two clusters) for all higher values. The regulariser Q₂(Θ) in Equation (8) scales linearly with λ, and once it dominates the log-likelihood term Q₁(Θ), the M-step updates for μₖ and Σₖ are driven entirely by the graph-smoothness penalty rather than by the data. The result is cluster centres that are "smooth" over the graph but do not correspond to the true data distribution. This failure is a direct consequence of the absence of any automatic mechanism for balancing λ against the data likelihood — the paper itself simply sets λ=0.1 empirically for all datasets (Section "Compared Algorithms").

**Uniform high-dimensional data (Scenario 2):** Both LCGMM (31.0%) and GMM (29.5%) perform near the random-chance baseline of 25%. In this case the kNN graph connects geometrically close but semantically unrelated points, making the regulariser actively misleading. This connects directly to **Assumption 1 (Task 1.2)**: the manifold assumption is violated, the graph does not encode meaningful semantic proximity, and the regulariser provides zero benefit.

**Connecting to Task 1.2 assumptions:** The over-regularisation failure is inherent to the lack of automatic model selection (Assumption 3). The uniform-noise failure is a direct consequence of violating the manifold assumption (Assumption 1). Both failure modes were predictable from the paper's assumptions.

**Suggested modification:** To address the over-regularisation failure, the objective could incorporate an **automatic λ selection** via Bayesian Information Criterion (BIC) or a held-out validation set, similar to how the number of components K is often chosen in practice. Specifically, one could treat λ as a hyperparameter optimised over a grid on a small held-out subset, choosing the λ that maximises a BIC-penalised log-likelihood.
